PART 02：核心手术——循环一行不动，把硬编码换成查表
先把上篇的主循环请回来，盯住工具执行那两行：

In [ ]:
for block in tool_calls:
    print(f"$ {block.input['command']}")
    output = run_bash(block.input["command"])

问题就出在 run_bash 是硬编码的——写死了"任何工具都去调 bash 的执行器"。现在要加四个新工具，你当然可以写成这样：

In [ ]:
if block.name == "bash":
    output = run_bash(**block.input)
elif block.name == "read_file":
    output = run_read(**block.input)
elif block.name == "write_file":
    output = run_write(**block.input)
# ...每加一个工具,再加一个 elif

能跑，但丑。五个工具五条 elif，五十个工具五十条，这条链会越长越臃肿，而且每一段都在重复同一件事：拿名字找函数。

"拿名字找函数"，Python 里有个现成的数据结构就是专门干这个的——字典：

In [ ]:
TOOL_HANDLERS = {
    "bash":       run_bash,
    "read_file":  run_read,
    "write_file": run_write,
    "edit_file":  run_edit,
    "glob":       run_glob,
}

循环里那两行硬编码，换成两行查表：

In [ ]:
for block in tool_calls:
    handler = TOOL_HANDLERS.get(block.name)
    output = handler(**block.input) if handler else f"Unknown: {block.name}"
    results.append({
        "type": "tool_result",
        "tool_use_id": block.id,
        "content": output,
    })

手术到此结束。 while True 没动、tool_use 判断没动、账本追加没动、退出条件没动——上一篇拆的三个机制原封不动，变的只是"工具怎么找到自己的执行器"这一处。

一个字典，就是一块插座板
我喜欢把 TOOL_HANDLERS 想象成一块插座板：每个插座贴着工具名，插上什么电器，就有什么功能。主循环是墙里固定的电线，从来不问插座上插的是什么。

这里面还有个值得单独拎出来的细节：**block.input。

模型调工具时，给的是一段 JSON，比如 {"path": "a.py", "limit": 50}。而 run_read 的函数签名是 run_read(path, limit=None)。**block.input 一展开，JSON 的 key 直接变成函数的关键字参数——模型给的 JSON 和 Python 函数入参，天然对齐。你不需要写任何"参数翻译"代码，工具的 input_schema 定义成什么样，函数签名照着写就行。

加一个工具 = 在两个地方各加一行
现在，给这台 Agent 加新工具的完整流程，收敛成了两件事：

在 TOOLS 数组加一条 schema
——这是给模型看的，告诉它"你有这个能力、参数长什么样"
在 TOOL_HANDLERS 字典加一行映射
——这是给代码用的，工具名来了找谁执行
一个管"告示"，一个管"接线"，两个注册点各司其职。

但正因为是两个地方，就埋了一个新的坑：它们必须保持同步。如果你在 TOOLS 里告诉模型"你有 write_file"，却在 TOOL_HANDLERS 里忘了注册，会发生什么？模型兴冲冲申请调用，查表查到一个空——这就是上面代码里 if handler else f"Unknown: {block.name}" 的作用：把"没接线的插座"变成一句返回给模型的错误文本，而不是让整个程序当场崩溃。

注意这个处理方式的味道：错误不是抛给程序员看的，是塞回给模型看的。 模型收到 "Unknown: write_file"，下一圈会自己换 bash 绕路完成任务。又呼应了上篇那句话——报错是模型的眼睛。

这行查表，就是扩展性的种子
别小看这次替换。if-else 到字典的升级，表面上是从"五条分支"变成"五行映射"，实质是把加工具的成本从"改代码"降到了"填表格"——改代码要理解逻辑，填表格只需要守格式。

真实 Claude Code 里那个庞大的工具生态——读写编辑、搜索、子代理、MCP 服务器带进来的各种外部工具——底层就是这同一个模式：循环不变，注册表往里加条目。你后面接 MCP 插件时会看到，挂一个新 MCP 服务器，本质就是往这张表里动态塞进一批新映射，模型立刻就会用了。

心脏一次到位，之后只长器官，不动心脏。这是整个复刻系列最重要的架构基调，这一篇正式立起来了。